## Generate Training Dataset — Per-Subject EEG Preprocessing

Starting from the raw EXIST 2026 JSON, this notebook:

1. Removes ET and HR modalities. They are not used in this project.
2. We expand per-subject: each `(meme_id, subject_id)` pair becomes an independent row
3. Flattens EEG features to 80 columns (EXG_Channel_{N}_{Band}_power)
4. Processes annotation labels into hard (0/1) and soft (proportions) representations

Output: data/processed/train_base.parquet 


### Label encoding

| Column | Type | Rule |
|--------|------|------|
| sexist_hard | int 0/1 | Majority vote across 6 annotators |
| sexist_soft | float [0,1] | Proportion of YES votes |
| direct_hard | int 0/1 / NaN | Majority vote; NaN if meme is not sexist |
| direct_soft | float [0,1] / NaN | Proportion of DIRECT among sexist annotators |
| cat_hard_{name} | int 0/1 | 1 if ≥2 annotators selected the category |
| cat_soft_{name} | float [0,1] | Proportion of annotators selecting the category |

In [13]:
#Libraries
import json
import os
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

os.chdir("C:/Users/diego/Desktop/multimodal-eeg-spatial-gates")

In [14]:
#Paths
REPO_ROOT = "C:/Users/diego/Desktop/multimodal-eeg-spatial-gates"
DATA_DIR = os.path.join(REPO_ROOT, "data", "train")
OUTPUT_DIR = os.path.join(REPO_ROOT, "data", "processed")
os.makedirs(OUTPUT_DIR, exist_ok=True)

JSON_PATH = Path(DATA_DIR) / "EXIST2026_training.json"
OUTPUT_PATH = Path(OUTPUT_DIR) / "train_base.parquet"

print(f"JSON: {JSON_PATH}")
print(f"Output: {OUTPUT_PATH}")

JSON: C:\Users\diego\Desktop\multimodal-eeg-spatial-gates\data\train\EXIST2026_training.json
Output: C:\Users\diego\Desktop\multimodal-eeg-spatial-gates\data\processed\train_base.parquet


In [15]:
#Load Data
with open(JSON_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

df_raw = pd.DataFrame(list(raw_data.values()))
print(f"Raw dataset shape: {df_raw.shape}")
print(f"Columns: {df_raw.columns.tolist()}")
df_raw.head(2)

Raw dataset shape: (3984, 17)
Columns: ['id_EXIST', 'lang', 'text', 'meme', 'path_memes', 'number_annotators', 'annotators', 'gender_annotators', 'age_annotators', 'ethnicities_annotators', 'study_levels_annotators', 'countries_annotators', 'sensorial', 'labels_task2_1', 'labels_task2_2', 'labels_task2_3', 'split']


,id_EXIST,lang,text,meme,path_memes,number_annotators,annotators,gender_annotators,age_annotators,ethnicities_annotators,study_levels_annotators,countries_annotators,sensorial,labels_task2_1,labels_task2_2,labels_task2_3,split
0,110887,es,A VECES QUISIERAIR/ALZUMBA www.facebook.com/Oi...,110887.jpeg,memes/110887.jpeg,6,"[Annotator_193, Annotator_194, Annotator_195, ...","[F, F, F, M, M, M]","[18-22, 23-45, 46+, 46+, 18-22, 23-45]","[White or Caucasian, Multiracial, White or Cau...","[High school degree or equivalent, Master’s de...","[Sweden, United States, Spain, Portugal, Spain...","{'users': ['ES1', 'ES2', 'ES3'], 'modalities':...","[YES, YES, YES, YES, YES, YES]","[JUDGEMENTAL, DIRECT, DIRECT, DIRECT, DIRECT, ...","[[IDEOLOGICAL-INEQUALITY, OBJECTIFICATION], [M...",TRAIN-MEME_ES
1,110466,es,Se necesita cuidadora para adulto mayor.... fo...,110466.jpeg,memes/110466.jpeg,6,"[Annotator_103, Annotator_104, Annotator_105, ...","[F, F, F, M, M, M]","[18-22, 23-45, 46+, 46+, 18-22, 23-45]","[White or Caucasian, White or Caucasian, Hispa...","[Bachelor’s degree, High school degree or equi...","[Spain, Portugal, Spain, Canada, United States...","{'users': ['ES1', 'ES2', 'ES3'], 'modalities':...","[YES, NO, YES, YES, NO, NO]","[DIRECT, -, DIRECT, DIRECT, -, -]","[[STEREOTYPING-DOMINANCE], [-], [STEREOTYPING-...",TRAIN-MEME_ES


In [16]:
#Features 

# EEG feature
BANDS     = ["Delta", "Theta", "Alpha", "Beta", "Gamma"]
N_CHANNELS = 16
EEG_COLS  = [f"EXG_Channel_{ch}_{band}_power" for ch in range(N_CHANNELS) for band in BANDS]


#Sexism-Categories
TASK23_CLASSES = [
    "IDEOLOGICAL-INEQUALITY",
    "STEREOTYPING-DOMINANCE",
    "OBJECTIFICATION",
    "SEXUAL-VIOLENCE",
    "MISOGYNY-NON-SEXUAL-VIOLENCE",
]

TASK23_SHORT = {
    "IDEOLOGICAL-INEQUALITY":"ideological_inequality",
    "STEREOTYPING-DOMINANCE":"stereotyping_dominance",
    "OBJECTIFICATION":"objectification",
    "SEXUAL-VIOLENCE":"sexual_violence",
    "MISOGYNY-NON-SEXUAL-VIOLENCE": "misogyny_nsv",
}

print(f"EEG columns ({len(EEG_COLS)}): {EEG_COLS[:5]} ... {EEG_COLS[-3:]}")
print(f"Task 2.3 classes: {TASK23_CLASSES}")

EEG columns (80): ['EXG_Channel_0_Delta_power', 'EXG_Channel_0_Theta_power', 'EXG_Channel_0_Alpha_power', 'EXG_Channel_0_Beta_power', 'EXG_Channel_0_Gamma_power'] ... ['EXG_Channel_15_Alpha_power', 'EXG_Channel_15_Beta_power', 'EXG_Channel_15_Gamma_power']
Task 2.3 classes: ['IDEOLOGICAL-INEQUALITY', 'STEREOTYPING-DOMINANCE', 'OBJECTIFICATION', 'SEXUAL-VIOLENCE', 'MISOGYNY-NON-SEXUAL-VIOLENCE']


In [17]:
#Helper Functions
def _valid(labels, exclude):
    return [x for x in labels if x not in exclude]

# Task 2.1 (SEXISM BINARY)
def process_task21(labels):
    """
    Returns:
        hard  (int 0/1 or NaN): majority vote
        soft  (float):proportion of YES votes
        valid (bool):whether a clear majority exists
    """
    valid = _valid(labels, {"UNKNOWN"})
    if not valid:
        return np.nan, np.nan, False

    counts = Counter(valid)
    yes = counts.get("YES", 0)
    no = counts.get("NO",  0)
    total = yes + no

    soft = yes / total
    valid_flag = True
    hard  = 1 if yes >= no else 0  

    return hard, soft, valid_flag


# Task 2.2 (Intetion: Direct or Judgmental)
def process_task22(labels):
    """
    Returns:
        hard  (int 0/1 or NaN): 1=DIRECT, 0=JUDGEMENTAL; NaN if no sexist annotators
        soft  (float or NaN):   proportion of DIRECT votes among sexist annotators
        valid (bool):           False if meme has no sexist annotators
    """
    valid = _valid(labels, {"UNKNOWN", "-"})
    if not valid:
        return np.nan, np.nan, False

    counts = Counter(valid)
    direct = counts.get("DIRECT",0)
    judg = counts.get("JUDGEMENTAL",0)
    total = direct + judg

    soft = direct / total
    hard = 1 if direct >= judg else 0   # if tie - Direct

    return hard, soft, True


#Task 2.3 (Categories - Multilabel)
def process_task23(labels_task23):
    """
    Returns:
        hard_dict  ({class: 0/1}): 1 if ≥ threshold annotators selected the category
        soft_dict  ({class: float}): proportion of annotators selecting each category
        valid      (bool): True if at least one category has a hard label of 1
    """
    valid_annotators = []
    for ann in labels_task23:
        if not isinstance(ann, list):
            continue
        filtered = [x for x in ann if x not in {"UNKNOWN", "-"}]
        valid_annotators.append(filtered)

    n = len(valid_annotators)
    if n == 0:
        nan_dict = {c: np.nan for c in TASK23_CLASSES}
        return nan_dict, nan_dict, False

    counts = Counter()
    for ann in valid_annotators:
        for c in ann:
            counts[c] += 1

    threshold = n // 2 + 1
    hard_dict = {c: int(counts.get(c, 0) >= threshold) for c in TASK23_CLASSES}
    soft_dict = {c: counts.get(c, 0) / n for c in TASK23_CLASSES}
    valid = any(hard_dict.values())

    return hard_dict, soft_dict, valid


#Extract EEG per meme
def extract_eeg_by_user(sensorial):
    """
    Returns dict {subject_id: {EXG_Channel_N_Band_power: float}}
    Only includes subjects that have EEG data.
    """
    if not isinstance(sensorial, dict):
        return {}
    return (
        sensorial
        .get("modalities", {})
        .get("EEG", {})
        .get("by_user", {}))

##  Per-subject expansion

For each meme, we iterate over all subjects that have EEG data and create one row per (meme_id, subject_id).  Labels are meme-level and ET and HR are dropped entirely.

In [ ]:
rows = []

for _, meme in df_raw.iterrows():

    #Labels
    s_hard, s_soft, s_valid = process_task21(meme["labels_task2_1"])
    d_hard, d_soft, d_valid = process_task22(meme["labels_task2_2"])
    c_hard, c_soft, c_valid = process_task23(meme["labels_task2_3"])


    #Hierarchical masking: If a meme is not sexist, we set the other labels to NaN or False
    meme_is_sexist = (s_hard == 1)
    if not meme_is_sexist:
        d_hard, d_soft, d_valid = np.nan, np.nan, False
        c_hard = {c: np.nan for c in TASK23_CLASSES}
        c_soft = {c: np.nan for c in TASK23_CLASSES}
        c_valid = False

    # Extract EEG
    eeg_by_user = extract_eeg_by_user(meme["sensorial"])
    n_eeg_subjects = len(eeg_by_user)

    for subject_id, eeg_dict in eeg_by_user.items():

        # We flatten the 80 EEG Features. NaN if a featuresi is missing.
        eeg_flat = {col: eeg_dict.get(col, np.nan) for col in EEG_COLS}

        row = {
            #We create the dataframe
            "meme_id": meme["id_EXIST"],
            "subject_id": subject_id,
            "lang": meme["lang"],
            "text": meme["text"],
            "image_file": meme["meme"],
            "split": "Train",
            "n_eeg_subjects": n_eeg_subjects
            ,
            #EEG
            **eeg_flat,

            #Task 2.1 (Sexism)
            "sexist_hard":  s_hard,
            "sexist_soft":  s_soft,    
            "sexist_valid": s_valid,

            #Task 2.2 (Direct)
            "direct_hard":  d_hard,    # 1=DIRECT, 0=JUDGEMENTAL
            "direct_soft":  d_soft,    
            "direct_valid": d_valid,

            # Task 2.3 (Hard)
            **{f"cat_hard_{TASK23_SHORT[c]}": c_hard[c] for c in TASK23_CLASSES},
            
            #Task 2.3 (Soft)
            **{f"cat_soft_{TASK23_SHORT[c]}": c_soft[c] for c in TASK23_CLASSES},

            "categories_valid": c_valid,
        }

        rows.append(row)


train_base = pd.DataFrame(rows)
print(f"Shape: {train_base.shape}")
print(f"Columns ({len(train_base.columns)}): {train_base.columns.tolist()[:10]} ...")
train_base.head(5)

Shape: (8090, 104)
Columns (104): ['meme_id', 'subject_id', 'lang', 'text', 'image_file', 'split', 'n_eeg_subjects', 'EXG_Channel_0_Delta_power', 'EXG_Channel_0_Theta_power', 'EXG_Channel_0_Alpha_power'] ...


,meme_id,subject_id,lang,text,image_file,split,n_eeg_subjects,EXG_Channel_0_Delta_power,EXG_Channel_0_Theta_power,EXG_Channel_0_Alpha_power,...,cat_hard_stereotyping_dominance,cat_hard_objectification,cat_hard_sexual_violence,cat_hard_misogyny_nsv,cat_soft_ideological_inequality,cat_soft_stereotyping_dominance,cat_soft_objectification,cat_soft_sexual_violence,cat_soft_misogyny_nsv,categories_valid
0,110887,ES1,es,A VECES QUISIERAIR/ALZUMBA www.facebook.com/Oi...,110887.jpeg,Train,2,-0.5379,-0.6630,-0.6211,...,0.0,1.0,0.0,0.0,0.166667,0.166667,0.666667,0.000000,0.500000,True
1,110887,ES3,es,A VECES QUISIERAIR/ALZUMBA www.facebook.com/Oi...,110887.jpeg,Train,2,-0.3520,-0.3575,-0.0700,...,0.0,1.0,0.0,0.0,0.166667,0.166667,0.666667,0.000000,0.500000,True
2,110466,ES1,es,Se necesita cuidadora para adulto mayor.... fo...,110466.jpeg,Train,2,0.4666,0.0987,0.7818,...,0.0,0.0,0.0,0.0,0.166667,0.500000,0.166667,0.333333,0.166667,False
3,110466,ES2,es,Se necesita cuidadora para adulto mayor.... fo...,110466.jpeg,Train,2,0.3235,0.4141,1.1124,...,0.0,0.0,0.0,0.0,0.166667,0.500000,0.166667,0.333333,0.166667,False
4,111269,ES1,es,tomboy como son el anime y manga pre to tomboy...,111269.jpeg,Train,2,0.6395,0.4390,0.3173,...,0.0,1.0,0.0,0.0,0.000000,0.166667,0.666667,0.166667,0.166667,True


In [19]:
# Some quick statistics
print(f"Total rows (meme x subject): {len(train_base):,}")
print(f"Unique memes: {train_base['meme_id'].nunique():,}")
print(f"Unique subjects {train_base['subject_id'].nunique()}")
print(f"Languages: {train_base['lang'].value_counts().to_dict()}")

print("\nEEG subjects per meme:")
print(train_base.groupby('meme_id')['subject_id'].count().value_counts().sort_index())

print("\nSubject ID counts:")
print(train_base['subject_id'].value_counts())

Total rows (meme x subject): 8,090
Unique memes: 3,984
Unique subjects 12
Languages: {'en': 4059, 'es': 4031}

EEG subjects per meme:
subject_id
2    3862
3     122
Name: count, dtype: int64

Subject ID counts:
subject_id
ES1    1075
EN6    1052
EN1    1031
EN5    1027
ES5    1025
ES3    1015
EN3     826
ES4     820
ES2      87
EN7      67
ES8      58
EN2       7
Name: count, dtype: int64


In [20]:
# Label Distributions

print("Task 2.1: Sexist (hard):")
print(train_base['sexist_hard'].value_counts(dropna=False))
print(f"  Proportion sexist: {train_base['sexist_hard'].mean():.3f}")

print("\nTask 2.1: Sexist (soft, mean proportion YES):")
print(train_base['sexist_soft'].describe().round(3))

print("\nTask 2.2: Direct (hard, sexist memes only):")
print(train_base.loc[train_base['direct_valid'], 'direct_hard'].value_counts(dropna=False))

print("\nTask 2.3: Category hard label counts (among sexist memes):")
cat_hard_cols = [f"cat_hard_{s}" for s in TASK23_SHORT.values()]
print(train_base[cat_hard_cols].sum().rename(lambda x: x.replace("cat_hard_", "")))

Task 2.1: Sexist (hard):
sexist_hard
1    5321
0    2769
Name: count, dtype: int64
  Proportion sexist: 0.658

Task 2.1: Sexist (soft, mean proportion YES):
count    8090.000
mean        0.556
std         0.314
min         0.000
25%         0.333
50%         0.667
75%         0.833
max         1.000
Name: sexist_soft, dtype: float64

Task 2.2: Direct (hard, sexist memes only):
direct_hard
1.0    4076
0.0    1245
Name: count, dtype: int64

Task 2.3: Category hard label counts (among sexist memes):
ideological_inequality    571.0
stereotyping_dominance    571.0
objectification           712.0
sexual_violence           290.0
misogyny_nsv               94.0
dtype: float64


In [21]:
train_base.head()

,meme_id,subject_id,lang,text,image_file,split,n_eeg_subjects,EXG_Channel_0_Delta_power,EXG_Channel_0_Theta_power,EXG_Channel_0_Alpha_power,...,cat_hard_stereotyping_dominance,cat_hard_objectification,cat_hard_sexual_violence,cat_hard_misogyny_nsv,cat_soft_ideological_inequality,cat_soft_stereotyping_dominance,cat_soft_objectification,cat_soft_sexual_violence,cat_soft_misogyny_nsv,categories_valid
0,110887,ES1,es,A VECES QUISIERAIR/ALZUMBA www.facebook.com/Oi...,110887.jpeg,Train,2,-0.5379,-0.6630,-0.6211,...,0.0,1.0,0.0,0.0,0.166667,0.166667,0.666667,0.000000,0.500000,True
1,110887,ES3,es,A VECES QUISIERAIR/ALZUMBA www.facebook.com/Oi...,110887.jpeg,Train,2,-0.3520,-0.3575,-0.0700,...,0.0,1.0,0.0,0.0,0.166667,0.166667,0.666667,0.000000,0.500000,True
2,110466,ES1,es,Se necesita cuidadora para adulto mayor.... fo...,110466.jpeg,Train,2,0.4666,0.0987,0.7818,...,0.0,0.0,0.0,0.0,0.166667,0.500000,0.166667,0.333333,0.166667,False
3,110466,ES2,es,Se necesita cuidadora para adulto mayor.... fo...,110466.jpeg,Train,2,0.3235,0.4141,1.1124,...,0.0,0.0,0.0,0.0,0.166667,0.500000,0.166667,0.333333,0.166667,False
4,111269,ES1,es,tomboy como son el anime y manga pre to tomboy...,111269.jpeg,Train,2,0.6395,0.4390,0.3173,...,0.0,1.0,0.0,0.0,0.000000,0.166667,0.666667,0.166667,0.166667,True


## Save to parquet

In [22]:
train_base.to_parquet(OUTPUT_PATH, index=False)
print(f"Saved  to {OUTPUT_PATH}")
print(f"Size: {OUTPUT_PATH.stat().st_size / 1024:.1f} KB")

Saved  to C:\Users\diego\Desktop\multimodal-eeg-spatial-gates\data\processed\train_base.parquet
Size: 4299.0 KB
